In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

In [10]:
df = pd.read_csv('../data/feautred.csv')
np.random.seed(42)

In [11]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'loan_term_months', 'employees_count',
       'business_type', 'is_rev_line_of_loan', 'loan_amount_rub',
       'loan_status', 'is_franchise', 'industry_sector', 'business_age_months',
       'okved_risk_level', 'authorized_capital', 'revenue_last_year',
       'revenue_growth_rate', 'net_profit', 'has_tax_arrears',
       'has_blocked_accounts', 'bki_score', 'current_debt_load',
       'max_delinquency_days', 'active_credits_count', 'fssp_debt_amount',
       'active_lawsuits_count', 'loan_purpose', 'ownership_structure',
       'business_scale'],
      dtype='str')

In [18]:
df.head()

,Unnamed: 0.1,Unnamed: 0,loan_term_months,employees_count,business_type,is_rev_line_of_loan,loan_amount_rub,loan_status,is_franchise,industry_sector,...,has_blocked_accounts,bki_score,current_debt_load,max_delinquency_days,active_credits_count,fssp_debt_amount,active_lawsuits_count,loan_purpose,ownership_structure,business_scale
0,0,0,84,1.609438,2,0,15.408819,1,0,45,...,0,782,6.999399e+06,0,4,0.0,0,working_capital,2,1
1,1,1,60,1.098612,2,0,15.003354,1,0,72,...,0,800,2.626400e+06,0,1,5000.0,0,investment,2,1
2,2,2,180,2.079442,1,0,16.973957,1,0,62,...,0,724,5.968333e+07,0,2,0.0,0,working_capital,1,1
3,3,3,60,1.098612,1,0,14.869823,1,0,unknown,...,0,833,4.564599e+06,0,1,0.0,2,working_capital,2,1
4,4,4,240,2.708050,1,0,16.748197,1,0,unknown,...,1,672,2.465180e+07,0,0,0.0,0,working_capital,2,1


#### Масштабируем признаки

In [20]:
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "loan_term_months",
    "employees_count",
    "loan_amount_rub",
    "business_age_months",
    "authorized_capital",
    "revenue_last_year",
    "revenue_growth_rate",
    "net_profit",
    "bki_score",
    "current_debt_load",
    "max_delinquency_days",
    "active_credits_count",
    "fssp_debt_amount",
    "active_lawsuits_count",
]

X_numeric_source = df[numeric_cols].copy()

# Эти признаки ещё не логарифмированы.
log_cols = [
    "authorized_capital",
    "revenue_last_year",
    "current_debt_load",
    "fssp_debt_amount",
]

X_numeric_source[log_cols] = np.log1p(X_numeric_source[log_cols])

X_numeric_source["net_profit"] = (
    np.sign(X_numeric_source["net_profit"])
    * np.log1p(np.abs(X_numeric_source["net_profit"]))
)

scaler = StandardScaler()

X_numeric = pd.DataFrame(
    scaler.fit_transform(X_numeric_source),
    columns=numeric_cols,
    index=df.index,
)

#### Кодируем(BinaryEncoder) признаки

In [27]:
from category_encoders import BinaryEncoder

categorical_cols = [
    "business_type",
    "industry_sector",
    "loan_purpose",
    "ownership_structure",
    "business_scale",
    "okved_risk_level",
]

encoder = BinaryEncoder(
    cols=categorical_cols,
    handle_unknown="value",
    handle_missing="value",
    return_df=True,
)

X_encoded = encoder.fit_transform(
    df[categorical_cols].astype(str)
)

Объединяем признаки

In [28]:
binary_cols = [
    "is_rev_line_of_loan",
    "is_franchise",
    "has_tax_arrears",
    "has_blocked_accounts",
]

X_cluster = pd.concat(
    [
        X_numeric,
        X_encoded,
        df[binary_cols],
    ],
    axis=1,
).astype("float32")

print(X_cluster.shape)

(871007, 23)


#### обучаем MiniBatchKMeans

In [32]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

# Одна и та же выборка только для оценки качества.
X_eval = X_cluster.sample(n=2000, random_state=42)

results = []
best_score = -np.inf

for k in [10, 15, 20]:
    model = MiniBatchKMeans(
        n_clusters=k,
        batch_size=8192,
        n_init=10,
        max_iter=200,
        random_state=42,
    )

    # Обучение на всех данных.
    model.fit(X_cluster)

    score = silhouette_score(
        X_eval,
        model.predict(X_eval),
    )

    results.append({
        "k": k,
        "silhouette": score,
    })

    # Сохраняем лучший обученный вариант в памяти.
    if score > best_score:
        best_score = score
        cluster_model = model

    print(f"k={k}: silhouette={score:.4f}")

cluster_results = pd.DataFrame(results).sort_values(
    "silhouette",
    ascending=False,
)

print("\nСравнение вариантов:")
print(cluster_results.to_string(index=False))

# Результат лучшей по silhouette модели для каждой строки.
df_clusters = df.copy()
df_clusters["cluster"] = cluster_model.predict(X_cluster)

print("\nВыбрано кластеров:", cluster_model.n_clusters)

k=10: silhouette=0.1029
k=15: silhouette=0.0848
k=20: silhouette=0.0802

Сравнение вариантов:
 k  silhouette
10    0.102891
15    0.084759
20    0.080177

Выбрано кластеров: 10


In [35]:
# Размеры кластеров.
cluster_sizes = (
    df_clusters.groupby("cluster")
    .size()
    .rename("rows")
    .to_frame()
)

cluster_sizes["share_pct"] = (
    cluster_sizes["rows"] / len(df_clusters) * 100
).round(2)

print("Размеры кластеров:")
print(cluster_sizes.to_string())

# Возвращаем сумму и число сотрудников в исходную шкалу.
profile_data = df_clusters[
    ["cluster"] + numeric_cols
].copy()

profile_data["loan_amount_rub"] = np.expm1(
    profile_data["loan_amount_rub"]
)

profile_data["employees_count"] = np.rint(
    np.expm1(profile_data["employees_count"])
)

print("\nМедианы числовых признаков:")
print(
    profile_data.groupby("cluster")[numeric_cols]
    .median()
    .round(2)
    .to_string()
)

print("\nСамые частые категории:")
print(
    df_clusters.groupby("cluster")[categorical_cols]
    .agg(lambda x: x.value_counts().idxmax())
    .to_string()
)

print("\nДоля единиц в бинарных признаках:")
print(
    df_clusters.groupby("cluster")[binary_cols]
    .mean()
    .round(3)
    .to_string()
)

Размеры кластеров:
           rows  share_pct
cluster                   
0        108761      12.49
1        112279      12.89
2         97379      11.18
3         85973       9.87
4         54261       6.23
5         68340       7.85
6         96828      11.12
7         75086       8.62
8         87093      10.00
9         85007       9.76

Медианы числовых признаков:
         loan_term_months  employees_count  loan_amount_rub
cluster                                                    
0                   240.0              5.0       23124000.0
1                    84.0              4.0        4614714.0
2                    60.0              2.0        2870000.0
3                    84.0              2.0        5330000.0
4                   240.0             27.0       34440000.0
5                   120.0              6.0       18368000.0
6                    84.0             25.0       13612000.0
7                    84.0              1.0         842222.0
8                    84.0   

#### Сохраняем модель и encoder

In [37]:
import joblib
from pathlib import Path

artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

bundle = {
    "model": cluster_model,
    "scaler": scaler,
    "encoder": encoder,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "binary_cols": binary_cols,
    "feature_columns": X_cluster.columns.tolist(),
    "log1p_cols": [
        "authorized_capital",
        "revenue_last_year",
        "current_debt_load",
        "fssp_debt_amount",
    ],
    "signed_log1p_cols": ["net_profit"],
    "already_log1p_cols": [
        "employees_count",
        "loan_amount_rub",
    ],
}

joblib.dump(
    bundle,
    artifacts_dir / "clustering_bundle.joblib",
    compress=3,
)

print("Модель и трансформеры сохранены")

Модель и трансформеры сохранены
